In [1]:
import xarray as xr, numpy as np, pandas as pd, os, glob
import dask.dataframe as dd
from dask.distributed import Client

In [2]:
client = Client(n_workers=14, threads_per_worker=1, memory_limit='4GB')
client

/g/data/xp65/public/apps/med_conda/envs/analysis3-25.07/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 46249 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/46249/status,
Dashboard: /proxy/46249/status,Workers: 14
Total threads: 14,Total memory: 52.15 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:36249,Workers: 0
Dashboard: /proxy/46249/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:36581,Total threads: 1
Dashboard: /proxy/44875/status,Memory: 3.73 GiB
Nanny: tcp://127.0.0.1:46331,


In [3]:
ehf_fpath = '/scratch/ng72/ms5578/ehf_netcdf'
nmap_path = '/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess'
write_path = '/scratch/ng72/ms5578/time_series'
netcdf_files = glob.glob(os.path.join(ehf_fpath, "*.nc")) 

In [4]:
gen_df = pd.read_csv(f"{nmap_path}/gen_info.csv")

In [5]:
template_ds = xr.open_dataset(netcdf_files[0], engine='netcdf4')
lat_grid = template_ds['lat'].values
lon_grid = template_ds['lon'].values
gen_lats = gen_df['lat'].values
gen_lons = gen_df['lon'].values
lat_indices = np.abs(lat_grid[:, None] - gen_lats).argmin(axis=0)
lon_indices = np.abs(lon_grid[:, None] - gen_lons).argmin(axis=0)

In [6]:
isel_dict = {
    'lat': xr.DataArray(lat_indices, dims='points'),
    'lon': xr.DataArray(lon_indices, dims='points'),
}

all_ddfs = []

for file in netcdf_files:
    ds = xr.open_dataset(file, engine='netcdf4', chunks='auto')
    sub_ds = ds.isel(lat=isel_dict['lat'], lon=isel_dict['lon'])
    sub_ds = sub_ds.assign_coords(DUID=('points', gen_df['DUID'].values))
    df = sub_ds[['tas','EHF_val', 'HW_EHF_avg', 'HW_EHF_peak', 'EHF_flag','tas_3d_avg','tas_3d_peak']].to_dask_dataframe().reset_index()
    all_ddfs.append(df)


full_ddf = dd.concat(all_ddfs)

In [7]:
def clean_and_process(df):
    df['time'] = dd.to_datetime(df['time'])
    df = df.replace(1.000000e+20, np.nan)
    df = df.drop(columns=['height', 'crs','points','index'], errors='ignore')
    df = df.sort_values(by=['DUID', 'time'])
    return df

In [8]:
def number_heatwave_days(df):
    df = df.sort_values(by='time')
    is_hw = df['EHF_flag'] == 1
    event_id = (is_hw != is_hw.shift()).cumsum()
    df['event_group'] = np.where(is_hw, event_id, pd.NA)
    df['HW_event_day'] = df.groupby('event_group').cumcount() + 1
    df.loc[df['event_group'].isna(), 'HW_event_day'] = pd.NA
    return df

In [9]:
df = clean_and_process(full_ddf).compute()
df = df.groupby('DUID', group_keys=False).apply(number_heatwave_days)

/jobfs/145004933.gadi-pbs/ipykernel_3734609/2015718923.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('DUID', group_keys=False).apply(number_heatwave_days)


In [10]:
df.to_csv(f"{write_path}/gen_hw_status.csv", index=False)
gen_df.to_csv('/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/nmap.csv', index=False)